In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"

df = pd.read_csv(
    DATA_RAW / "all_matches.csv",
    dtype={"ball": str, "season": str, "match_id": str},
    low_memory=False,
)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Innings values:", sorted(df["innings"].unique().tolist()))

Rows: 295732
Columns: 27
Innings values: [1, 2, 3, 4, 5, 6]


In [3]:
before = len(df)
df = df[df["innings"].isin([1, 2])].copy()

print(f"Dropped {before - len(df)} super-over rows")
print("New shape:", df.shape)
print("Remaining innings:", sorted(df["innings"].unique().tolist()))

Dropped 175 super-over rows
New shape: (295557, 27)
Remaining innings: [1, 2]


In [4]:
parts = df["ball"].str.split(".", expand=True)
df["over"]         = parts[0].astype(int)   # 0-based: over 0 = 1st over
df["ball_in_over"] = parts[1].astype(int)   # delivery number within the over

print("Over range:", df["over"].min(), "→", df["over"].max())
print("Ball-in-over range:", df["ball_in_over"].min(), "→", df["ball_in_over"].max())
print(df[["ball", "over", "ball_in_over"]].head(3).to_string())

Over range: 0 → 19
Ball-in-over range: 1 → 11
  ball  over  ball_in_over
0  0.1     0             1
1  0.2     0             2
2  0.3     0             3


In [5]:
# Legal ball = neither a wide nor a no-ball
df["is_legal"] = df["wides"].isna() & df["noballs"].isna()

# Total runs on this delivery = runs off the bat + all extras
df["total_runs"] = df["runs_off_bat"] + df["extras"].fillna(0)

# A wicket that reduces the batting side (retired hurt does NOT)
df["is_wicket"] = df["wicket_type"].notna() & (df["wicket_type"] != "retired hurt")

# Wickets the bowler is credited with (excludes run-outs, obstruction, retired)
BOWLER_WICKETS = {
    "bowled", "caught", "caught and bowled", "lbw",
    "stumped", "hit wicket",
}
df["is_bowler_wicket"] = df["wicket_type"].isin(BOWLER_WICKETS)

print("Rows where wides is NaN (no wide):", df["wides"].isna().sum())
print("Total runs on first 5 balls:", df["total_runs"].head(5).tolist())
print("Total wickets in dataset:", int(df["is_wicket"].sum()))
print("Bowler-credited wickets:", int(df["is_bowler_wicket"].sum()))

Rows where wides is NaN (no wide): 285686
Total runs on first 5 balls: [0, 0, 4, 0, 2]
Total wickets in dataset: 14655
Bowler-credited wickets: 13460


In [6]:
# First: see what team names actually exist in your data
print("Teams before mapping:")
print(sorted(df["batting_team"].unique().tolist()))

Teams before mapping:
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Delhi Daredevils', 'Gujarat Lions', 'Gujarat Titans', 'Kings XI Punjab', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Rising Pune Supergiants', 'Royal Challengers Bangalore', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


In [7]:
TEAM_MAP = {
    "Delhi Daredevils":            "Delhi Capitals",
    "Kings XI Punjab":             "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants":     "Rising Pune Supergiant",
}

df["batting_team"] = df["batting_team"].replace(TEAM_MAP)
df["bowling_team"] = df["bowling_team"].replace(TEAM_MAP)

print("Unique teams after mapping:", df["batting_team"].nunique())
print("Teams after mapping:")
print(sorted(df["batting_team"].unique().tolist()))

Unique teams after mapping: 15
Teams after mapping:
['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiant', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


In [8]:
print("Total unique venues:", df["venue"].nunique())
print("\nTop 30 venues by matches:")
print(df["venue"].value_counts().head(30).to_string())

Total unique venues: 60

Top 30 venues by matches:
venue
Eden Gardens                                                             17988
Wankhede Stadium                                                         17584
M Chinnaswamy Stadium                                                    14895
Wankhede Stadium, Mumbai                                                 14280
Feroz Shah Kotla                                                         13950
MA Chidambaram Stadium, Chepauk                                          11751
Rajiv Gandhi International Stadium, Uppal                                11591
Dubai International Cricket Stadium                                      11187
Sawai Mansingh Stadium                                                   11150
MA Chidambaram Stadium, Chepauk, Chennai                                  9949
Narendra Modi Stadium, Ahmedabad                                          9778
Punjab Cricket Association Stadium, Mohali                                

In [9]:
df["venue"] = df["venue"].str.split(",").str[0].str.strip()

print("Unique venues after stripping city suffix:", df["venue"].nunique())
print(df["venue"].value_counts().head(20).to_string())

Unique venues after stripping city suffix: 42
venue
Wankhede Stadium                                                31864
Eden Gardens                                                    24936
MA Chidambaram Stadium                                          23830
Rajiv Gandhi International Stadium                              21403
M Chinnaswamy Stadium                                           20622
Sawai Mansingh Stadium                                          16174
Feroz Shah Kotla                                                13950
Dubai International Cricket Stadium                             11187
Arun Jaitley Stadium                                            10594
Narendra Modi Stadium                                            9778
Dr DY Patil Sports Academy                                       8898
Maharashtra Cricket Association Stadium                          8414
Punjab Cricket Association Stadium                               8266
Sheikh Zayed Stadium                  

In [10]:
VENUE_MAP = {
    "M.Chinnaswamy Stadium":              "M Chinnaswamy Stadium",
    "MA Chidambaram Stadium, Chepauk":    "MA Chidambaram Stadium",
    "Arun Jaitley Stadium":               "Arun Jaitley Stadium",   # keep, but folded in below
    "Feroz Shah Kotla":                   "Arun Jaitley Stadium",   # renamed 2019
    "Subrata Roy Sahara Stadium":         "Maharashtra Cricket Association Stadium",  # renamed 2013
}

df["venue"] = df["venue"].replace(VENUE_MAP)

print("Final unique venues:", df["venue"].nunique())
print(df["venue"].value_counts().to_string())

Final unique venues: 39
venue
Wankhede Stadium                                                31864
Eden Gardens                                                    24936
Arun Jaitley Stadium                                            24544
M Chinnaswamy Stadium                                           24116
MA Chidambaram Stadium                                          23830
Rajiv Gandhi International Stadium                              21403
Sawai Mansingh Stadium                                          16174
Maharashtra Cricket Association Stadium                         12239
Dubai International Cricket Stadium                             11187
Narendra Modi Stadium                                            9778
Dr DY Patil Sports Academy                                       8898
Punjab Cricket Association Stadium                               8266
Sheikh Zayed Stadium                                             6906
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium

In [11]:
"Sardar Patel Stadium": "Narendra Modi Stadium",

SyntaxError: illegal target for annotation (3630516499.py, line 1)

In [12]:
print(df["venue"].value_counts().tail(30).to_string())

venue
Narendra Modi Stadium                                           9778
Dr DY Patil Sports Academy                                      8898
Punjab Cricket Association Stadium                              8266
Sheikh Zayed Stadium                                            6906
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium    6900
Sharjah Cricket Stadium                                         6672
Brabourne Stadium                                               6526
Punjab Cricket Association IS Bindra Stadium                    6210
Himachal Pradesh Cricket Association Stadium                    4461
Maharaja Yadavindra Singh International Cricket Stadium         4049
Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium             4000
Kingsmead                                                       3643
Sardar Patel Stadium                                            2882
SuperSport Park                                                 2866
Saurashtra Cricket Associati

In [13]:
VENUE_MAP = {
    # Punjab: same Mohali ground, full name is "IS Bindra"
    "Punjab Cricket Association Stadium":        "Punjab Cricket Association IS Bindra Stadium",

    # Abu Dhabi: same ground, two spellings
    "Zayed Cricket Stadium":                     "Sheikh Zayed Stadium",

    # Ahmedabad: renamed Sardar Patel → Narendra Modi in 2021
    "Sardar Patel Stadium":                      "Narendra Modi Stadium",
}

df["venue"] = df["venue"].replace(VENUE_MAP)

print("Final unique venues:", df["venue"].nunique())
print("\nTop 25 venues:")
print(df["venue"].value_counts().head(25).to_string())

Final unique venues: 36

Top 25 venues:
venue
Wankhede Stadium                                                31864
Eden Gardens                                                    24936
Arun Jaitley Stadium                                            24544
M Chinnaswamy Stadium                                           24116
MA Chidambaram Stadium                                          23830
Rajiv Gandhi International Stadium                              21403
Sawai Mansingh Stadium                                          16174
Punjab Cricket Association IS Bindra Stadium                    14476
Narendra Modi Stadium                                           12660
Maharashtra Cricket Association Stadium                         12239
Dubai International Cricket Stadium                             11187
Dr DY Patil Sports Academy                                       8898
Sheikh Zayed Stadium                                             8780
Bharat Ratna Shri Atal Bihari Vajpayee Ekana

In [14]:
# Any venue with under 200 deliveries is suspicious (likely a typo)
rare = df["venue"].value_counts()
print("Venues with < 200 deliveries:")
print(rare[rare < 200].to_string())

Venues with < 200 deliveries:
Series([], )


In [15]:
# Phase: over is 0-based, so over 0-5 = overs 1-6, etc.
df["phase"] = pd.cut(
    df["over"],
    bins=[-1, 5, 14, 19],
    labels=["Powerplay", "Middle", "Death"],
)

# Cumulative state computed WITHIN each innings
g = df.groupby(["match_id", "innings"], sort=False)

df["runs_so_far"]        = g["total_runs"].cumsum()
df["wickets_so_far"]     = g["is_wicket"].cumsum()
df["legal_balls_so_far"] = g["is_legal"].cumsum()
df["balls_remaining"]    = 120 - df["legal_balls_so_far"]
df["wickets_in_hand"]    = 10 - df["wickets_so_far"]

print("Phase counts:")
print(df["phase"].value_counts().to_string())
print()
print("Null phase rows:", df["phase"].isna().sum())
print("Sample of new columns:")
print(df[["match_id", "innings", "over", "phase",
          "runs_so_far", "wickets_so_far",
          "balls_remaining", "wickets_in_hand"]].head(8).to_string())

Phase counts:
phase
Middle       135655
Powerplay     92729
Death         67173

Null phase rows: 0
Sample of new columns:
  match_id  innings  over      phase  runs_so_far  wickets_so_far  balls_remaining  wickets_in_hand
0  1082591        1     0  Powerplay            0               0              119               10
1  1082591        1     0  Powerplay            0               0              118               10
2  1082591        1     0  Powerplay            4               0              117               10
3  1082591        1     0  Powerplay            4               0              116               10
4  1082591        1     0  Powerplay            6               0              116               10
5  1082591        1     0  Powerplay            6               0              115               10
6  1082591        1     0  Powerplay            7               0              114               10
7  1082591        1     1  Powerplay            8               0            

In [16]:
# For completed innings, last row should show balls_remaining near 0
# For all-out innings, it should show balls_remaining > 0
last_rows = df.groupby(["match_id", "innings"], sort=False).tail(1)

print("Distribution of balls_remaining at innings end:")
print(last_rows["balls_remaining"].describe().to_string())
print()
print("Innings ending exactly at 120 legal balls:",
      int((last_rows["balls_remaining"] == 0).sum()))
print("Innings ending early (all out):",
      int((last_rows["balls_remaining"] > 0).sum()))
print("Innings ending in impossible state (balls_remaining < 0):",
      int((last_rows["balls_remaining"] < 0).sum()))

Distribution of balls_remaining at innings end:
count    2480.000000
mean        5.296371
std        12.614928
min        -1.000000
25%         0.000000
50%         0.000000
75%         5.000000
max       114.000000

Innings ending exactly at 120 legal balls: 1611
Innings ending early (all out): 868
Innings ending in impossible state (balls_remaining < 0): 1


In [17]:
bad = df[df["balls_remaining"] < 0]
print(bad[["match_id", "innings", "start_date", "venue",
           "over", "ball", "bowler", "striker",
           "legal_balls_so_far", "balls_remaining"]].to_string())

       match_id  innings  start_date                   venue  over  ball   bowler      striker  legal_balls_so_far  balls_remaining
198224   419155        1  2010-04-15  MA Chidambaram Stadium    19  19.6  A Nehra  S Badrinath                 121               -1


In [18]:
# Clip balls_remaining at 0 — the only affected row is a single final
# delivery of one innings with a recorded 21st ball in the 20th over.
df["balls_remaining"] = (120 - df["legal_balls_so_far"]).clip(lower=0)

# Confirm
print("Min balls_remaining:", df["balls_remaining"].min())   # → 0
print("Rows with balls_remaining < 0:", int((df["balls_remaining"] < 0).sum()))  # → 0

Min balls_remaining: 0
Rows with balls_remaining < 0: 0


In [19]:
from pathlib import Path

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "cleaned_deliveries.csv"
df.to_csv(out_path, index=False)

print("Saved:", out_path.resolve())
print("Final shape:", df.shape)
print("Years covered:", int(df["season_year"].min()), "→", int(df["season_year"].max())
      if "season_year" in df.columns
      else "season_year column missing")

Saved: C:\Users\pa1ku\OneDrive\Desktop\IPL_Data_Analytics\data\processed\cleaned_deliveries.csv
Final shape: (295557, 39)


KeyError: 'season_year'

In [20]:
df["season_year"] = pd.to_datetime(df["start_date"]).dt.year
print("Years:", int(df["season_year"].min()), "→", int(df["season_year"].max()))

Years: 2008 → 2026


In [21]:
chk = pd.read_csv("../data/processed/cleaned_deliveries.csv",
                  dtype={"ball": str, "match_id": str}, low_memory=False)

print("Rows:", len(chk))
print("Matches:", chk["match_id"].nunique())
print("Innings values:", sorted(chk["innings"].unique().tolist()))
print("Null phase:", int(chk["phase"].isna().sum()))
print("Null season_year:", int(chk["season_year"].isna().sum()))
print("Null balls_remaining:", int(chk["balls_remaining"].isna().sum()))
print("Min balls_remaining:", int(chk["balls_remaining"].min()))

Rows: 295557
Matches: 1243
Innings values: [1, 2]
Null phase: 0


KeyError: 'season_year'

In [22]:
print("season_year in df?", "season_year" in df.columns)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

season_year in df? True
Shape: (295557, 40)
Columns: ['match_id', 'season', 'start_date', 'venue', 'innings', 'ball', 'actual_delivery', 'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler', 'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty', 'non_boundary', 'wicket_type', 'player_dismissed', 'other_wicket_type', 'other_player_dismissed', 'fielder_1', 'fielder_2', 'fielder_3', 'over', 'ball_in_over', 'is_legal', 'total_runs', 'is_wicket', 'is_bowler_wicket', 'phase', 'runs_so_far', 'wickets_so_far', 'legal_balls_so_far', 'balls_remaining', 'wickets_in_hand', 'season_year']


In [23]:
df["season_year"] = pd.to_datetime(df["start_date"]).dt.year

print("Years covered:", int(df["season_year"].min()), "→", int(df["season_year"].max()))
print("Null season_year rows:", int(df["season_year"].isna().sum()))

Years covered: 2008 → 2026
Null season_year rows: 0


In [24]:
from pathlib import Path

out_path = Path("../data/processed/cleaned_deliveries.csv")
df.to_csv(out_path, index=False)

print("Re-saved:", out_path.resolve())
print("Final shape:", df.shape)

Re-saved: C:\Users\pa1ku\OneDrive\Desktop\IPL_Data_Analytics\data\processed\cleaned_deliveries.csv
Final shape: (295557, 40)


In [25]:
chk = pd.read_csv("../data/processed/cleaned_deliveries.csv",
                  dtype={"ball": str, "match_id": str}, low_memory=False)

print("Rows:", len(chk))
print("Matches:", chk["match_id"].nunique())
print("Innings values:", sorted(chk["innings"].unique().tolist()))
print("Null phase:", int(chk["phase"].isna().sum()))
print("Null season_year:", int(chk["season_year"].isna().sum()))
print("Null balls_remaining:", int(chk["balls_remaining"].isna().sum()))
print("Min balls_remaining:", int(chk["balls_remaining"].min()))

Rows: 295557
Matches: 1243
Innings values: [1, 2]
Null phase: 0
Null season_year: 0
Null balls_remaining: 0
Min balls_remaining: 0


In [26]:
required = [
    "over", "ball_in_over", "is_legal", "total_runs",
    "is_wicket", "is_bowler_wicket", "phase",
    "runs_so_far", "wickets_so_far", "legal_balls_so_far",
    "balls_remaining", "wickets_in_hand", "season_year",
]
missing = [c for c in required if c not in df.columns]
print("Missing helper columns:", missing if missing else "none — all present ✅")

Missing helper columns: none — all present ✅


In [27]:
from pathlib import Path

out_path = Path("../data/processed/cleaned_deliveries.csv")
df.to_csv(out_path, index=False)

print("Saved:", out_path.resolve())
print("Shape:", df.shape)
print("Has season_year on disk? Yes (we just wrote it)")

Saved: C:\Users\pa1ku\OneDrive\Desktop\IPL_Data_Analytics\data\processed\cleaned_deliveries.csv
Shape: (295557, 40)
Has season_year on disk? Yes (we just wrote it)


In [28]:
chk = pd.read_csv("../data/processed/cleaned_deliveries.csv",
                  dtype={"ball": str, "match_id": str}, low_memory=False)

print("Rows:", len(chk))
print("Matches:", chk["match_id"].nunique())
print("Innings values:", sorted(chk["innings"].unique().tolist()))
print("Null phase:", int(chk["phase"].isna().sum()))
print("Null season_year:", int(chk["season_year"].isna().sum()))
print("Null balls_remaining:", int(chk["balls_remaining"].isna().sum()))
print("Min balls_remaining:", int(chk["balls_remaining"].min()))

Rows: 295557
Matches: 1243
Innings values: [1, 2]
Null phase: 0
Null season_year: 0
Null balls_remaining: 0
Min balls_remaining: 0
